In [4]:
"""
build_assets_colab.py
---------------------
Colab 최적화 버전. 아래 순서로 실행하세요.

  1. Colab 에 이 파일 업로드
  2. CSV 2개 업로드 (코드 실행 시 팝업)
  3. python build_assets_colab.py
  4. 완료 후 models.joblib / df_full.pkl / short_signal_pipeline.py 자동 다운로드

개선 사항
  - GBR → LightGBM quantile (속도 10배+, n_jobs=-1)
  - easing / ret 타깃 학습 루프 밖에서 사전 계산
  - 모델마다 체크포인트 저장 (중간에 꺼져도 재개 가능)
  - 단계별 RAM 사용량 출력
"""

# ── 0. 패키지 확인 ────────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import lightgbm, psutil
except ImportError:
    print("필요 패키지 설치 중...")
    pip_install("lightgbm", "psutil")

# ── 1. Import ─────────────────────────────────────────────────────
import time
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import psutil
from sklearn.metrics import roc_auc_score

# ── 2. 공용 상수 ──────────────────────────────────────────────────
RANDOM_STATE   = 42
HORIZON_DAYS   = 5
VAL_SIZE_DAYS  = 20
TEST_SIZE_DAYS = 20
Q              = 0.7
MIN_HIST_DAYS  = 30

FEATURE_COLS = [
    "log_mktcap", "log_price", "is_large_cap",
    "ret_1d", "ret_5d", "ret_20d", "vol_5d", "vol_20d", "intraday_vol", "price_ma20_gap",
    "turnover", "log_trdval", "abnormal_vol", "trdval_ma5_over_ma20", "ret5_x_abnormal_vol",
    "short_ratio_ma5", "short_ratio_ma10", "short_ratio_lag1",
    "short_ratio_chg_1d", "short_ratio_std5",
    "balance_ratio_chg_5d", "balance_ratio_ma5", "balance_ratio_lag1", "short_qty_chg5",
    "short_ratio_pct_vs_own_hist", "balance_ratio_pct_vs_own_hist",
]

LGBM_CLF_BASE = dict(
    random_state=RANDOM_STATE, verbose=-1, n_jobs=-1,
    n_estimators=1000, learning_rate=0.02, num_leaves=127,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
)

LGBM_QNT_BASE = dict(
    objective="quantile",
    random_state=RANDOM_STATE, verbose=-1, n_jobs=-1,
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
)

# ── 3. 유틸리티 ───────────────────────────────────────────────────
def mem():
    v = psutil.virtual_memory()
    return f"RAM {v.used/1e9:.1f}/{v.total/1e9:.1f}GB ({v.percent:.0f}%)"

def checkpoint(models, path="models_checkpoint.joblib"):
    joblib.dump(models, path)
    print(f"  💾 체크포인트 저장 ({len(models)}개) → {path}")

def upload_csvs():
    """Colab 환경에서만 파일 업로드 팝업 실행"""
    try:
        from google.colab import files
        print("📂 CSV 파일 2개를 업로드하세요:")
        print("  ① df_price_20250401_20260331_with_60d_buffer.csv")
        print("  ② df_short_merged_20250304_20260331.csv\n")
        uploaded = files.upload()
        for name in uploaded:
            print(f"  ✅ {name}  ({len(uploaded[name])/1e6:.1f} MB)")
    except ImportError:
        print("ℹ️  로컬 환경 — CSV 파일이 현재 디렉터리에 있어야 합니다.")

def download_outputs():
    """Colab 환경에서만 자동 다운로드 실행"""
    try:
        from google.colab import files
        print("\n📥 결과물 다운로드...")
        for f in ["models.joblib", "df_full.pkl", "short_signal_pipeline.py"]:
            files.download(f)
            print(f"  ⬇️  {f}")
    except ImportError:
        print("ℹ️  로컬 환경 — 현재 디렉터리에 파일이 저장되었습니다.")

# ── 4. 데이터 로드 ────────────────────────────────────────────────
def clean_numeric(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ""), errors="coerce")

def load_data():
    price  = pd.read_csv("df_price_20250401_20260331_with_60d_buffer.csv")
    merged = pd.read_csv("df_short_merged_20250304_20260331.csv")

    for d in (price, merged):
        d["ISU_CD"] = d["ISU_CD"].astype(str).str.zfill(6)
        d["date"]   = pd.to_datetime(d["date"])

    price[["TDD_OPNPRC","TDD_HGPRC","TDD_LWPRC","TDD_CLSPRC","ACC_TRDVOL","CHG_RT"]] = \
        price[["TDD_OPNPRC","TDD_HGPRC","TDD_LWPRC","TDD_CLSPRC","ACC_TRDVOL","CHG_RT"]].apply(clean_numeric)
    merged[["short_qty","acc_trdvol","short_ratio","balance_ratio","MKTCAP","LIST_SHRS"]] = \
        merged[["short_qty","acc_trdvol","short_ratio","balance_ratio","MKTCAP","LIST_SHRS"]].apply(clean_numeric)

    df = price.merge(
        merged[["ISU_CD","date","short_qty","short_ratio","balance_ratio","MKTCAP","LIST_SHRS","acc_trdvol"]],
        on=["ISU_CD","date"], how="inner",
    )
    return df.sort_values(["ISU_CD","date"]).reset_index(drop=True)

# ── 5. 피처 엔지니어링 ────────────────────────────────────────────
def add_features(df):
    g = df.groupby("ISU_CD")

    df["log_mktcap"]   = np.log(df["MKTCAP"].clip(lower=1))
    df["log_price"]    = np.log(df["TDD_CLSPRC"].clip(lower=1))
    df["is_large_cap"] = (df["MKTCAP"] >= g["MKTCAP"].transform("median")).astype(int)

    df["ret_1d"]  = g["TDD_CLSPRC"].pct_change(1)
    df["ret_5d"]  = g["TDD_CLSPRC"].pct_change(5)
    df["ret_20d"] = g["TDD_CLSPRC"].pct_change(20)
    df["vol_5d"]  = g["ret_1d"].transform(lambda x: x.rolling(5).std())
    df["vol_20d"] = g["ret_1d"].transform(lambda x: x.rolling(20).std())
    df["intraday_vol"]   = (df["TDD_HGPRC"] - df["TDD_LWPRC"]) / df["TDD_CLSPRC"]
    ma20 = g["TDD_CLSPRC"].transform(lambda x: x.rolling(20).mean())
    df["price_ma20_gap"] = (df["TDD_CLSPRC"] - ma20) / ma20

    trdval = df["ACC_TRDVOL"] * df["TDD_CLSPRC"]
    df["turnover"]             = df["ACC_TRDVOL"] / df["LIST_SHRS"]
    df["log_trdval"]           = np.log(trdval.clip(lower=1))
    df["abnormal_vol"]         = df["ACC_TRDVOL"] / g["ACC_TRDVOL"].transform(lambda x: x.rolling(20).mean())
    trdval_g                   = trdval.groupby(df["ISU_CD"])
    df["trdval_ma5_over_ma20"] = trdval_g.transform(lambda x: x.rolling(5).mean()) / \
                                 trdval_g.transform(lambda x: x.rolling(20).mean())
    df["ret5_x_abnormal_vol"]  = df["ret_5d"] * df["abnormal_vol"]

    df["short_ratio_ma5"]      = g["short_ratio"].transform(lambda x: x.rolling(5).mean())
    df["short_ratio_ma10"]     = g["short_ratio"].transform(lambda x: x.rolling(10).mean())
    df["short_ratio_lag1"]     = g["short_ratio"].shift(1)
    df["short_ratio_chg_1d"]   = g["short_ratio"].diff(1)
    df["short_ratio_std5"]     = g["short_ratio"].transform(lambda x: x.rolling(5).std())
    df["balance_ratio_chg_5d"] = g["balance_ratio"].diff(5)
    df["balance_ratio_ma5"]    = g["balance_ratio"].transform(lambda x: x.rolling(5).mean())
    df["balance_ratio_lag1"]   = g["balance_ratio"].shift(1)
    df["short_qty_chg5"]       = g["short_qty"].pct_change(5).replace([np.inf, -np.inf], np.nan)

    df["short_ratio_pct_vs_own_hist"] = g["short_ratio"].transform(
        lambda x: x.shift(1).expanding(min_periods=MIN_HIST_DAYS).rank(pct=True)
    )
    df["balance_ratio_pct_vs_own_hist"] = g["balance_ratio"].transform(
        lambda x: x.shift(1).expanding(min_periods=MIN_HIST_DAYS).rank(pct=True)
    )
    return df

# ── 6. 타깃 & 사전 계산 ──────────────────────────────────────────
def add_target(df):
    g = df.groupby("ISU_CD")
    df["future_short_ratio_median_h5"] = g["short_ratio"].transform(
        lambda x: x.shift(-1).rolling(HORIZON_DAYS).median()
    )
    df["future_balance_ratio_h5"] = g["balance_ratio"].shift(-HORIZON_DAYS)
    df["target_score_h5"] = (
        0.7 * df["future_short_ratio_median_h5"] + 0.3 * df["future_balance_ratio_h5"]
    )
    df["own_hist_q70"] = df.groupby("ISU_CD")["target_score_h5"].transform(
        lambda x: x.shift(1).expanding(min_periods=MIN_HIST_DAYS).quantile(Q)
    )
    df["target"] = (
        (df["target_score_h5"] >= df["own_hist_q70"]) & (df["target_score_h5"] > 0)
    ).astype(int)
    return df

def add_horizon_targets(df):
    """easing / ret 타깃을 루프 밖에서 한번에 사전 계산 (핵심 개선)"""
    g = df.groupby("ISU_CD")
    for h in range(1, HORIZON_DAYS + 1):
        future_pct          = g["short_ratio_pct_vs_own_hist"].shift(-h)
        col_e               = f"easing_target_{h}"
        df[col_e]           = (future_pct < df["short_ratio_pct_vs_own_hist"]).astype(float)
        df.loc[future_pct.isna(), col_e] = np.nan
        df[f"future_ret_h{h}"] = g["TDD_CLSPRC"].shift(-h) / df["TDD_CLSPRC"] - 1
    return df

# ── 7. 시계열 분할 ────────────────────────────────────────────────
def time_split(df):
    clean        = df.dropna(subset=FEATURE_COLS + ["target"]).copy()
    dates        = sorted(clean["date"].unique())
    test_dates   = dates[-TEST_SIZE_DAYS:]
    val_dates    = dates[-(TEST_SIZE_DAYS + VAL_SIZE_DAYS):-TEST_SIZE_DAYS]
    train_dates  = dates[:-(TEST_SIZE_DAYS + VAL_SIZE_DAYS)]
    return (
        clean[clean["date"].isin(train_dates)],
        clean[clean["date"].isin(val_dates)],
        clean[clean["date"].isin(test_dates)],
    )

# ── 8. 모델 학습 헬퍼 ────────────────────────────────────────────
def fit_lgbm_clf(X_tr, y_tr, X_vl, y_vl, class_weight=None):
    m = lgb.LGBMClassifier(**{**LGBM_CLF_BASE, "class_weight": class_weight})
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, y_vl)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(80, verbose=False)],
    )
    return m

def fit_lgbm_quantile(X_tr, y_tr, X_vl, y_vl, alpha):
    m = lgb.LGBMRegressor(**{**LGBM_QNT_BASE, "alpha": alpha})
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, y_vl)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    return m

# ── 9. MAIN ───────────────────────────────────────────────────────
if __name__ == "__main__":

    # Step 0 — 파일 업로드
    upload_csvs()

    # Step 1 — 데이터
    print("\n[1/4] 데이터 로드 & 전처리")
    t0 = time.time()
    df = load_data()
    print(f"  로드: {len(df):,}행  |  {mem()}")
    df = add_features(df)
    print(f"  피처: 완료  |  {mem()}")
    df = add_target(df)
    df = add_horizon_targets(df)
    print(f"  타깃: 완료  |  {mem()}  ({time.time()-t0:.1f}s)")

    # Step 2 — df_full.pkl
    df.to_pickle("df_full.pkl")
    print("\n[2/4] df_full.pkl 저장 완료")

    # Step 3 — 분할
    train, val, test = time_split(df)
    X_tr, y_tr = train[FEATURE_COLS], train["target"]
    X_vl, y_vl = val[FEATURE_COLS],   val["target"]
    X_te, y_te = test[FEATURE_COLS],  test["target"]
    train_dates = set(train["date"])
    val_dates   = set(val["date"])
    test_dates  = set(test["date"])

    pos_w = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    print(f"\n[3/4] 모델 학습  |  train {train['date'].nunique()}일 / val {val['date'].nunique()}일 / test {test['date'].nunique()}일")
    print(f"  pos_weight={pos_w:.2f}")

    models = {}

    # (A) pressure
    print("\n  [A] pressure 모델")
    t0 = time.time()
    models["pressure"] = fit_lgbm_clf(X_tr, y_tr, X_vl, y_vl, class_weight={0: 1.0, 1: pos_w})
    auc = roc_auc_score(y_te, models["pressure"].predict_proba(X_te)[:, 1])
    print(f"      AUC={auc:.4f}  ({time.time()-t0:.1f}s)")
    checkpoint(models)

    # (B) easing D+1 ~ D+5
    print("\n  [B] easing 모델 (D+1 ~ D+5)")
    for h in range(1, HORIZON_DAYS + 1):
        col = f"easing_target_{h}"
        sub = df.dropna(subset=FEATURE_COLS + [col])
        tr  = sub[sub["date"].isin(train_dates)]
        vl  = sub[sub["date"].isin(val_dates)]
        te  = sub[sub["date"].isin(test_dates)]
        pw  = (tr[col] == 0).sum() / max((tr[col] == 1).sum(), 1)

        t0 = time.time()
        m  = fit_lgbm_clf(tr[FEATURE_COLS], tr[col], vl[FEATURE_COLS], vl[col],
                          class_weight={0: 1.0, 1: pw})
        auc = roc_auc_score(te[col], m.predict_proba(te[FEATURE_COLS])[:, 1])
        print(f"      D+{h}  AUC={auc:.4f}  ({time.time()-t0:.1f}s)  |  {mem()}")
        models[f"easing_{h}"] = m
        checkpoint(models)

    # (C) ret quantile (LightGBM으로 교체)
    print("\n  [C] ret quantile 모델 (LightGBM quantile, D+1 ~ D+5)")
    for h in range(1, HORIZON_DAYS + 1):
        ret_col = f"future_ret_h{h}"
        sub = df.dropna(subset=FEATURE_COLS + [ret_col])
        tr  = sub[sub["date"].isin(train_dates)]
        vl  = sub[sub["date"].isin(val_dates)]

        t0 = time.time()
        for alpha, tag in [(0.1, "q10"), (0.5, "q50"), (0.9, "q90")]:
            models[f"ret_{tag}_{h}"] = fit_lgbm_quantile(
                tr[FEATURE_COLS], tr[ret_col],
                vl[FEATURE_COLS], vl[ret_col],
                alpha=alpha,
            )
        print(f"      D+{h}  완료  ({time.time()-t0:.1f}s)  |  {mem()}")
        checkpoint(models)

    # Step 4 — 최종 저장
    joblib.dump(models, "models.joblib")
    print(f"\n[4/4] models.joblib 저장 완료 | 총 {len(models)}개 모델")

    # short_signal_pipeline.py
    with open("short_signal_pipeline.py", "w", encoding="utf-8") as f:
        f.write(f'"""\nshort_signal_pipeline.py  -  app.py 가 import 하는 공용 상수 모듈\n"""\n'
                f"FEATURE_COLS = {FEATURE_COLS!r}\n\n"
                f"RANDOM_STATE  = {RANDOM_STATE}\n"
                f"HORIZON_DAYS  = {HORIZON_DAYS}\n"
                f"MIN_HIST_DAYS = {MIN_HIST_DAYS}\n"
                f"Q             = {Q}\n")
    print("     short_signal_pipeline.py 생성 완료")

    # 다운로드
    download_outputs()

    print("\n" + "=" * 50)
    print("완료! 로컬에서 대시보드를 실행하세요:")
    print("  streamlit run app.py")
    print("=" * 50)

ℹ️  로컬 환경 — CSV 파일이 현재 디렉터리에 있어야 합니다.

[1/4] 데이터 로드 & 전처리
  로드: 248,125행  |  RAM 2.7/8.6GB (82%)
  피처: 완료  |  RAM 2.8/8.6GB (82%)
  타깃: 완료  |  RAM 2.8/8.6GB (83%)  (4.7s)

[2/4] df_full.pkl 저장 완료

[3/4] 모델 학습  |  train 193일 / val 20일 / test 20일
  pos_weight=1.70

  [A] pressure 모델
      AUC=0.8493  (24.2s)
  💾 체크포인트 저장 (1개) → models_checkpoint.joblib

  [B] easing 모델 (D+1 ~ D+5)
      D+1  AUC=1.0000  (2.4s)  |  RAM 2.8/8.6GB (82%)
  💾 체크포인트 저장 (2개) → models_checkpoint.joblib
      D+2  AUC=0.8017  (8.2s)  |  RAM 2.7/8.6GB (84%)
  💾 체크포인트 저장 (3개) → models_checkpoint.joblib
      D+3  AUC=0.7934  (10.2s)  |  RAM 2.7/8.6GB (83%)
  💾 체크포인트 저장 (4개) → models_checkpoint.joblib
      D+4  AUC=0.7950  (11.7s)  |  RAM 2.7/8.6GB (83%)
  💾 체크포인트 저장 (5개) → models_checkpoint.joblib
      D+5  AUC=0.7992  (11.6s)  |  RAM 2.7/8.6GB (83%)
  💾 체크포인트 저장 (6개) → models_checkpoint.joblib

  [C] ret quantile 모델 (LightGBM quantile, D+1 ~ D+5)
      D+1  완료  (4.3s)  |  RAM 2.7/8.6GB (84%)
  💾 체크포인트 저장 (9개) → 

In [2]:
import threading, subprocess
def run():
    subprocess.run(["streamlit", "run", "app.py",
                    "--server.port", "8501",
                    "--server.headless", "true"])
t = threading.Thread(target=run, daemon=True)
t.start()

import time; time.sleep(3)
print("http://localhost:8501")




  You can now view your Streamlit app in your browser.

  Network URL: http://192.168.35.13:8501
  External URL: http://39.122.219.166:8501

http://localhost:8501


2026-06-14 23:46:05.355 Uncaught app exception
Traceback (most recent call last):
  File "/Users/leesehyun/miniforge3/envs/krc/lib/python3.8/site-packages/streamlit/runtime/scriptrunner/script_runner.py", line 534, in _run_script
    exec(code, module.__dict__)
  File "/Users/leesehyun/Desktop/XAI/app.py", line 88, in <module>
    pressure_prob = models["pressure"].predict_proba(row[FEATURE_COLS])[0, 1]
KeyError: 'pressure'


In [1]:
import subprocess
subprocess.run(["pkill", "-f", "streamlit"])

CompletedProcess(args=['pkill', '-f', 'streamlit'], returncode=1)